In [1]:
%cd "C:\Users\Public\Downloads\Uni Lecture\Data Preprocessing\Final's Project\Flight Delay\src"

C:\Users\Public\Downloads\Uni Lecture\Data Preprocessing\Final's Project\Flight Delay\src


In [2]:
import pandas as pd
import numpy as np
import airportsdata
import requests
import time
from pathlib import Path
from tqdm import tqdm

from utils.utilities import check_schema, dtype_converter

In [3]:
'''
--LOAD DATA--
'''
# cols = ['origin', 'dest', 'crs_dep_time', 'year', 'month', 'day_of_month']

print('LOADING DATA...')
RAW_PATH = Path('../data/raw/pre_weather_crawl')
df_dt = pd.read_csv('../data/flight_data_2024_data_dictionary.csv')
df1 = pd.read_csv(RAW_PATH / 'flight_2024.csv', low_memory=False)
df2 = pd.read_csv(RAW_PATH / 'flight_2025.csv', low_memory=False)
print('DONE LOADING.')
print('--------------------------------')

dtypes = dict(zip(df_dt['column'], df_dt['dtype']))

check_schema(df1, df2)
print('CONVERTING DATA TYPES...')
df1 = dtype_converter(df1, dtypes)
df2 = dtype_converter(df2, dtypes)
df = pd.concat([df1, df2], axis=0)
print('DONE CONVERTING.')
print('|----------------|')

LOADING DATA...
DONE LOADING.
--------------------------------
CONVERTING DATA TYPES...
DONE CONVERTING.
|----------------|


In [4]:
df['hour'] = (df['crs_dep_time'] // 100).astype(np.int8)
df['minute'] = (df['crs_dep_time'] % 100).astype(np.int8)
df['crs_dep_time'] = df['crs_dep_time'].astype(str).str.zfill(4)

In [5]:
codes = df.origin.unique().tolist()

In [6]:
airports = airportsdata.load('IATA')
rows = []

for code in codes:
    if code in airports:
        a = airports[code]

        rows.append({
            'airport': code,
            'latitude': a['lat'],
            'longitude': a['lon'],
        })

airport_coor = pd.DataFrame(rows)
airport_coor.shape

(358, 3)

In [7]:
df['origin'] = df['origin'].str.strip().str.upper()
df['dest'] = df['dest'].str.strip().str.upper()
airport_coor['airport'] = airport_coor['airport'].str.strip().str.upper()

In [8]:
df['datetime'] = pd.to_datetime({
    'year': df['year'],
    'month': df['month'],
    'day': df['day_of_month'],
    'hour': df['hour'],
    'minute': df['minute']
})

df['weather_hour'] = df['datetime'].dt.floor('h')

In [9]:
airport_coor = pd.DataFrame({'airport': codes}).merge(
    airport_coor,
    on='airport',
    how='left'
)

airport_coor.dropna(inplace=True)
len(airport_coor)

358

In [10]:
def get_weather_meteo(url, lat, lon, start_date, end_date):
    params = {
        'latitude': lat,
        'longitude': lon,
        'start_date': start_date,
        'end_date': end_date,
        'hourly': [
            'temperature_2m', 'surface_pressure', 'relative_humidity_2m',
            "dew_point_2m", "wind_speed_10m", "wind_direction_10m",
            "cloud_cover", "weather_code"
        ],
        'timezone': 'UTC'
    }

    max_retries = 5
    delay = 10

    for attempt in tqdm(range(max_retries), leave=False):
        try:
            response = requests.get(url, params=params, timeout=60)

            if response.status_code == 200:
                return response.json()
            
            elif response.status_code == 429:
                print(f'\n Error code: 429 (Overloaded). Waiting for {delay} seconds'
                      f'before trying again (Attempt {attempt + 1}/{max_retries})...'
                )
                time.sleep(delay)
                delay *= 1.5

            else:
                print(f'\nError code: {response.status_code}')
                return None
            
        except requests.exceptions.RequestException as e:
            print(f'\nConnection Error/Timeout: {e}. Waiting for 5s before trying again...')
            time.sleep(5)

    print(f'\nFailed to request data at coordinate ({lat}, {lon})')
    return None

# Crawl Weather 2024

In [ ]:
weather_data = []

START_DATE = '2024-01-01'
END_DATE = '2024-12-31'
meteo_url = "https://archive-api.open-meteo.com/v1/archive"

for _, row in tqdm(airport_coor.iterrows(), desc='Airport', total=len(airport_coor)):
    try:
        airport = row['airport']

        result = get_weather_meteo(
            meteo_url,
            row['latitude'],
            row['longitude'],
            START_DATE,
            END_DATE
        )

        if result is None:
            continue

        hourly = result.get('hourly', {})

        weather_df = pd.DataFrame({
            'time': hourly.get('time', []),
            'temperature_2m': hourly.get('temperature_2m', []),
            'surface_pressure': hourly.get('surface_pressure', []),
            'relative_humidity_2m': hourly.get('relative_humidity_2m', []),
            'dew_point_2m': hourly.get('dew_point_2m', []),
            'wind_speed_10m': hourly.get('wind_speed_10m', []),
            'wind_direction_10m': hourly.get('wind_direction_10m', []),
            'cloud_cover': hourly.get('cloud_cover', []),
            'weather_code': hourly.get('weather_code', [])
        })

        weather_df['airport'] = airport
        weather_data.append(weather_df)

        time.sleep(2)

    except Exception as e:
        print(f'\nERROR at {airport}: {e}')

In [13]:
weather_df = pd.concat(
    weather_data,
    ignore_index=True
)

weather_df['weather_hour'] = pd.to_datetime(weather_df['time'])

In [ ]:
missed_airports = list(set(airport_coor['airport']) - set(weather_df['airport']))
missed_coor = airport_coor[airport_coor['airport'].isin(missed_airports)]

weather_data2 = []

for _, row in tqdm(missed_coor.iterrows(), desc='Airport', total=len(missed_coor)):
    try:
        airport = row['airport']

        result = get_weather_meteo(
            meteo_url,
            row['latitude'],
            row['longitude'],
            START_DATE,
            END_DATE
        )

        if result is None:
            continue

        hourly = result.get('hourly', {})

        weather_df2 = pd.DataFrame({
            'time': hourly.get('time', []),
            'temperature_2m': hourly.get('temperature_2m', []),
            'surface_pressure': hourly.get('surface_pressure', []),
            'relative_humidity_2m': hourly.get('relative_humidity_2m', []),
            'dew_point_2m': hourly.get('dew_point_2m', []),
            'wind_speed_10m': hourly.get('wind_speed_10m', []),
            'wind_direction_10m': hourly.get('wind_direction_10m', []),
            'cloud_cover': hourly.get('cloud_cover', []),
            'weather_code': hourly.get('weather_code', [])
        })

        weather_df2['airport'] = airport
        weather_data2.append(weather_df2)

        time.sleep(2)

    except Exception as e:
        print(f'\nERROR at {airport}: {e}')

In [ ]:
if len(weather_data2) > 0:
    weather_df2 = pd.concat(
        weather_data2,
        ignore_index=True
    )
    weather_df2['weather_hour'] = pd.to_datetime(weather_df2['time'])

    weather_df_final = pd.concat([weather_df, weather_df2])

else:
    weather_df_final = weather_df

weather_df_final.drop(columns=['time'], inplace=True, errors='ignore')

df_final = df1.merge(
    weather_df_final,
    on=['origin', 'weather_hour'],
    how='left'
)

df_final.to_csv('../data/raw/flight_24.csv', index=False)

# Crawl Weather 2025